# 01 - Exploratory Data Analysis (M5 subset: CA_1 / FOODS / 12 SKUs)

Goal: understand the demand patterns the agents must handle - level, weekly seasonality, SNAP and event effects, intermittency, censored demand (stockouts) and price changes. Run from the repo root or `notebooks/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.width", 160)

In [ ]:
sales = pd.read_parquet(ROOT / "data/processed/sales_subset.parquet")
future_cal = pd.read_parquet(ROOT / "data/processed/calendar_future.parquet")
actuals = pd.read_parquet(ROOT / "data/processed/actuals_future.parquet")
print(sales.shape, sales["date"].min().date(), "->", sales["date"].max().date())
sales.head()

## 1. Item profile

In [ ]:
profile = sales.groupby("item_id").agg(
    mean_units=("units", "mean"), std_units=("units", "std"), max_units=("units", "max"),
    zero_share=("units", lambda s: (s == 0).mean()), avg_price=("sell_price", "mean"),
).round(3)
profile["cv"] = (profile["std_units"] / profile["mean_units"]).round(2)
profile.sort_values("mean_units", ascending=False)

High-volume smooth items (CV < 0.6) sit next to intermittent ones (zero share around 30%) - the Forecast Agent must handle both, and the Risk Agent should be less confident on the latter.

In [ ]:
fig = px.line(sales, x="date", y="units", color="item_id", height=420,
              title="Daily unit sales (2 years)")
fig.show()

## 2. Weekly seasonality

In [ ]:
sales["dow"] = sales["date"].dt.day_name().str[:3]
order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
idx = (sales.groupby(["item_id", "dow"])["units"].mean()
       / sales.groupby("item_id")["units"].mean()).rename("index").reset_index()
weekly = idx.pivot(index="item_id", columns="dow", values="index")[order].round(2)
weekly

In [ ]:
px.imshow(weekly, color_continuous_scale="RdBu_r", origin="lower", aspect="auto",
          title="Day-of-week demand index (1 = average)").show()

Saturdays and Sundays run roughly 10-50% above the item average -> weekly seasonal terms (SARIMA period 7, Prophet weekly seasonality) are essential.

## 3. SNAP and event effects

In [ ]:
snap = sales.groupby(["item_id", "snap"])["units"].mean().unstack()
snap["snap_uplift_%"] = ((snap[1] / snap[0] - 1) * 100).round(1)
event = sales.assign(is_event=sales["event_name"] != "").groupby(
    ["item_id", "is_event"])["units"].mean().unstack()
event["event_effect_%"] = ((event[True] / event[False] - 1) * 100).round(1)
pd.concat([snap["snap_uplift_%"], event["event_effect_%"]], axis=1)

In [ ]:
top_events = (sales[sales["event_name"] != ""].groupby("event_name")["units"].mean()
              / sales["units"].mean() - 1).sort_values()
px.bar(top_events.mul(100).round(1), orientation="h", height=520,
       title="Average demand vs normal day by event (%)").show()

SNAP days lift demand for most items (about +5% on average, up to +16%) -> used as a Prophet regressor. Some events (e.g. Christmas, store closure) collapse demand -> modelled as holidays.

## 4. Censored demand (suspected stockouts) and data quality

In [ ]:
from src.data.cleaning import clean_item_sales

rows = []
for item, grp in sales.groupby("item_id"):
    clean, report = clean_item_sales(grp)
    rows.append({"item_id": item, "longest_zero_streak": report.longest_zero_streak,
                 "suspected_stockout_days": report.suspected_stockout_days,
                 "quality_score": round(report.score, 3)})
quality = pd.DataFrame(rows).set_index("item_id")
quality

In [ ]:
item = quality["suspected_stockout_days"].idxmax()
clean, _ = clean_item_sales(sales[sales["item_id"] == item])
fig = go.Figure()
fig.add_scatter(x=clean["date"], y=clean["units"], name="observed sales", line_color="#999")
so = clean[clean["is_suspected_stockout"]]
fig.add_scatter(x=so["date"], y=so["units"], mode="markers", name="suspected stockout",
                marker_color="crimson")
fig.update_layout(title=f"{item}: zero streaks on an active item = censored demand", height=380)
fig.show()

Long zero runs on items selling 30-60 units/day are almost certainly empty shelves, not zero demand. Training on them would bias forecasts down, so the cleaning step sets `y = NaN` for those days and the quality score drops.

## 5. Price

In [ ]:
px.line(sales, x="date", y="sell_price", color="item_id", height=380,
        title="Weekly shelf price").show()
sales.groupby("item_id")["sell_price"].agg(["min", "max", "nunique"])

## 6. Forecast window and hold-out

In [ ]:
print("Future calendar:", future_cal["date"].min().date(), "->",
      future_cal["date"].max().date(), "| events:",
      future_cal.loc[future_cal["event_name"] != "", "event_name"].unique().tolist())
actuals.groupby("item_id")["units"].sum().rename("units_next_28d").to_frame().T

## Takeaways for the agent design

1. Strong weekly seasonality and SNAP effects -> SARIMA(7) and Prophet with SNAP regressor and event holidays.
2. Heterogeneous items -> per-item model selection by backtest, plus a seasonal-naive baseline to measure skill.
3. Censored demand and intermittency -> data-quality score feeds the Risk Agent and the Critic's `data_quality` rule.
4. Spiky, event-driven series -> anomaly / level-shift detection and an uncertainty multiplier on safety stock.